<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Clustering: Agrupar las Fincas de la Cooperativa 🥔 (Para Dummies)
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 07
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Estadistica%20analisis%20y%20representacion%20de%20datos/07%20-%20Temas%20Avanzados%20en%20Data%20Analysis/Para%20Dummies/01_Clustering_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del [cuaderno 01 del Módulo 07 (Clustering)](../01_Clustering.ipynb). Imagina que eres el encargado de una **cooperativa de papa** con 150 fincas y nadie te dijo cuáles son pequeñas, medianas o grandes: tienes solo los **números** (hectáreas, rendimiento, altitud). **Agrupar** es **juntar las fincas que se parecen entre sí**, sin que nadie te diga las respuestas de antemano.

| En la cooperativa | En estadística |
|---|---|
| Juntar fincas parecidas en «montoncitos» | **Clustering** (agrupamiento) |
| Poner 3 banderas en el mapa y pegar cada finca a la bandera más cercana | **k-means** |
| Mover cada bandera al centro de su montoncito y repetir | Las **iteraciones** del algoritmo |
| ¿Hacemos 2, 3 o 6 montoncitos? | **Elegir k** (el número de grupos) |
| Medir una en metros y otra en hectáreas | El problema de la **escala** |

Al terminar podrás explicar:
1. **Cómo funciona k-means** con la imagen de las banderas que se mueven.
2. **Cómo se elige** cuántos grupos hacer (el «codo»).
3. Por qué hay que **igualar las escalas** y por qué una variable que **no tiene que ver** con los grupos puede estropearlo todo.

> 🔗 Si luego quieres ver **otros algoritmos (jerárquico, DBSCAN, mezclas), criterios formales y simulaciones**: [cuaderno estándar](../01_Clustering.ipynb). Taller de todo el módulo: [`../../homeworks/Para Dummies/07_Temas_Avanzados_Hands_On_Dummies.ipynb`](../../homeworks/Para%20Dummies/07_Temas_Avanzados_Hands_On_Dummies.ipynb).

---
## 1. Las Banderas que se Mueven 🚩

Inventamos 150 fincas de **tres tipos** (pequeñas, medianas y grandes) y **no se lo decimos** al algoritmo; solo le damos hectáreas y rendimiento. La receta de k-means son tres pasos que se repiten: **(1)** poner 3 banderas al azar sobre fincas, **(2)** pegar cada finca a la bandera más cercana, **(3)** mover cada bandera al centro de su grupo. Se repite hasta que las banderas ya no se mueven. Mira cómo cambia el mapa entre el inicio, un paso y el final.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

# 150 fincas INVENTADAS de tres tipos: pequenas, medianas y grandes (esto NO se lo diremos al algoritmo)
tipo = rng.integers(0, 3, 150)
hectareas = np.array([1.0, 4.0, 9.0])[tipo] + rng.normal(0, 0.8, 150)
rendimiento = np.array([8.0, 15.0, 22.0])[tipo] + rng.normal(0, 2.5, 150)     # toneladas por hectarea
altitud = rng.uniform(2400, 3200, 150)                                          # metros: no tiene que ver con el tipo
fincas = pd.DataFrame({"hectareas": hectareas, "rendimiento": rendimiento, "altitud": altitud})
X2 = fincas[["hectareas", "rendimiento"]].values
print(fincas.head(3).round(1).to_string())

def distancias(X, centros):                                    # que tan lejos queda cada finca de cada centro
    return np.sqrt(((X[:, None, :] - centros[None, :, :]) ** 2).sum(axis=2))

def agrupar(X, k, semilla=0, pasos=20):
    r = np.random.default_rng(semilla)
    centros = X[r.choice(len(X), k, replace=False)]            # 1) elegimos k fincas al azar como "centros"
    historia = [centros.copy()]
    for _ in range(pasos):
        grupo = distancias(X, centros).argmin(axis=1)          # 2) cada finca se pega al centro mas cercano
        nuevos = np.array([X[grupo == j].mean(axis=0) if np.any(grupo == j) else centros[j] for j in range(k)])
        if np.allclose(nuevos, centros):
            break
        centros = nuevos                                       # 3) cada centro se mueve al medio de su grupo
        historia.append(centros.copy())
    grupo = distancias(X, centros).argmin(axis=1)
    desorden = float(((X - centros[grupo]) ** 2).sum())        # que tan "regados" quedaron los grupos (menor = mejor)
    return grupo, centros, desorden, historia

# Probamos 10 arranques distintos y nos quedamos con el mejor
mejor = min((agrupar(X2, 3, semilla=s) for s in range(10)), key=lambda r: r[2])
grupo, centros, desorden, historia = mejor
print(f"Pasos que tardo el mejor arranque: {len(historia)} | desorden final: {desorden:.0f}")

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for a, paso, titulo in zip(ax, (0, 1, len(historia) - 1), ("Al empezar: 3 centros al azar", "Despues de 1 paso", "Al final")):
    asig = distancias(X2, historia[paso]).argmin(axis=1)
    a.scatter(X2[:, 0], X2[:, 1], c=[["#2563eb", "#dc2626", "#16a34a"][g] for g in asig], s=12)
    a.scatter(*historia[paso].T, marker="X", s=200, color="k"); a.set_title(titulo); a.set_xlabel("hectareas"); a.set_ylabel("rendimiento")
plt.tight_layout(); plt.show()


---
**Lección:** con banderas puestas al azar el mapa empieza mal repartido, pero en pocos pasos el mapa se ordena: el mejor de 10 arranques tardó solo **9 pasos** en quedar quieto. Probamos **10 arranques distintos** y nos quedamos con el de menor «desorden» porque **un mal arranque puede dar un mal resultado** (las banderas empiezan al azar). El gráfico final deja ver tres montoncitos claros: pequeñas abajo a la izquierda, grandes arriba a la derecha.

---
## 2. ¿Cuántos Montoncitos? y la Trampa de la Escala ⚖️

Para saber si el algoritmo acertó usamos algo que en la vida real **no tendríamos**: el tipo verdadero de cada finca. Medimos qué porcentaje de fincas quedó junto a las de su mismo tipo (la **pureza**). Para elegir cuántos grupos hacer, probamos varios y miramos el **«desorden»** (qué tan regadas quedan las fincas respecto a su bandera): **siempre baja** al poner más banderas, así que buscamos el **codo**, donde deja de bajar rápido. Después hacemos una **trampa**: agregamos la **altitud** en metros (números enormes, 2 400 a 3 200) junto a las hectáreas (números chicos, 0 a 11).

In [ ]:
def pureza(grupo, verdad):
    """Que porcentaje de fincas quedo en un grupo cuyo tipo mayoritario es el suyo."""
    return sum(np.bincount(verdad[grupo == g]).max() for g in np.unique(grupo)) / len(verdad)

print(f"Con hectareas y rendimiento (k = 3): {pureza(grupo, tipo):.0%} de las fincas quedaron con las de su mismo tipo")

# Cuantos grupos? Probamos varios k y miramos el "desorden": siempre baja, buscamos el codo
desordenes = {k: min(agrupar(X2, k, semilla=s)[2] for s in range(10)) for k in range(1, 8)}
print({k: round(v) for k, v in desordenes.items()})

# La trampa de la escala: agregamos la altitud (en metros, valores enormes frente a las hectareas)
X_raw = fincas[["hectareas", "rendimiento", "altitud"]].values
g_raw = min((agrupar(X_raw, 3, semilla=s) for s in range(10)), key=lambda r: r[2])[0]
X_est = (X_raw - X_raw.mean(axis=0)) / X_raw.std(axis=0)           # misma regla para todas: "cuantas desviaciones del promedio"
g_est = min((agrupar(X_est, 3, semilla=s) for s in range(10)), key=lambda r: r[2])[0]
print(f"Con altitud SIN igualar escalas: {pureza(g_raw, tipo):.0%} | igualando las escalas: {pureza(g_est, tipo):.0%}")

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(list(desordenes), list(desordenes.values()), "o-", color="#2563eb"); ax[0].set_xlabel("numero de grupos"); ax[0].set_title("El codo: el desorden deja de bajar rapido en 3")
ax[1].scatter(fincas.altitud, fincas.hectareas, c=[["#2563eb", "#dc2626", "#16a34a"][g] for g in g_raw], s=12)
ax[1].set_xlabel("altitud (m)"); ax[1].set_ylabel("hectareas"); ax[1].set_title("Sin igualar escalas, manda la altitud")
plt.tight_layout(); plt.show()


---
**Lección:** (1) Con hectáreas y rendimiento y 3 grupos, **93 %** de las fincas quedó con las de su mismo tipo. (2) **El codo:** el desorden baja de 7 028 (1 grupo) a 2 039 (2) y a 939 (3), y de ahí cae poco (711 con 4, 531 con 5): el «codo» está en **3**, que es el número real. (3) **La trampa de la escala:** al agregar la altitud, la pureza se desploma a **43 %**, porque en metros la altitud tiene números **enormes** y manda sobre todo, como si una báscula pesara en toneladas y otra en gramos. Al **igualar las escalas** (medir cada variable en «cuántas desviaciones se aleja del promedio») sube a **71 %**. (4) **Pero no llega al 93 %**: la altitud **no tiene nada que ver** con el tipo de finca y, aun con escalas iguales, mete ruido. Moraleja: **igualar escalas** y **no meter variables que no sirven**.

> ⚠️ Los grupos del ejemplo son **redondos y parecidos en tamaño**, que es lo que k-means maneja bien. Con grupos alargados o con formas raras (como dos medias lunas) k-means puede fallar; eso se muestra en el cuaderno estándar.

---
##### 🛠️ Práctica 1: Hectáreas más altitud

Con el DataFrame `fincas`, el vector `tipo` (tipo verdadero) y las funciones `agrupar` y `pureza` ya definidas:
1. Toma solo las columnas `hectareas` y `altitud` (`X_hq`) e **iguala sus escalas** restando el promedio y dividiendo entre la desviación (`X_hq_est`).
2. Agrupa en 3 con `agrupar` (prueba 10 arranques y quédate con el menor desorden, que es `r[2]`) tanto `X_hq` sin igualar como `X_hq_est`, y calcula la pureza de cada uno (`p_raw`, `p_est`).
3. Comprueba con `assert` que igualar las escalas **mejora** (`p_est > p_raw`).

In [ ]:
# Escribe tu código aquí

# X_hq = fincas[['hectareas', 'altitud']].values
# X_hq_est = ...
# g_hq_raw = min((agrupar(X_hq, 3, semilla=s) for s in range(10)), key=lambda r: r[2])[0]
# g_hq = ...
# p_raw, p_est = ..., ...

# assert p_est > p_raw


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
# Igualamos las escalas SOLO de hectareas y altitud y ponemos rendimiento fuera
X_hq = fincas[["hectareas", "altitud"]].values
X_hq_est = (X_hq - X_hq.mean(axis=0)) / X_hq.std(axis=0)
g_hq = min((agrupar(X_hq_est, 3, semilla=s) for s in range(10)), key=lambda r: r[2])[0]
g_hq_raw = min((agrupar(X_hq, 3, semilla=s) for s in range(10)), key=lambda r: r[2])[0]
p_est, p_raw = pureza(g_hq, tipo), pureza(g_hq_raw, tipo)
print(f"hectareas + altitud: sin igualar = {p_raw:.0%} | igualando = {p_est:.0%}")
assert p_est > p_raw
```
</details>

---

---
### 🧠 ¿Ya lo tienes claro?

1. Con la imagen de las banderas: ¿por qué conviene probar **varios arranques** y quedarse con el mejor, en vez de confiar en uno solo?
2. El «desorden» siempre baja al agregar grupos. Si hiciéramos **un grupo por finca**, el desorden sería cero. ¿Por qué eso **no** sirve como agrupación útil, y cómo ayuda el «codo» a decidir?
3. Tienes las variables *edad del productor* (años) e *ingreso mensual* (pesos). Si agrupas **sin igualar escalas**, ¿cuál mandaría? ¿Qué harías antes de agrupar?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos (Edición Didáctica Para Dummies)</i>
  </p>
</div>